# APS360 — Transcript Classifier: Colab Training

> **Before running:** Enable GPU via `Runtime → Change runtime type → T4 GPU`

## 1. Clone repo & install dependencies

In [ ]:
import os

REPO_URL = "https://github.com/romeluis/APS360-TranscriptClassifier.git"
REPO_DIR = "/content/APS360-TranscriptClassifier"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull
    print("Repo already cloned — pulled latest changes.")

%cd {REPO_DIR}

In [ ]:
# Python packages
!pip install -q -r requirements.txt

# WeasyPrint system libs — required to render HTML → PDF for training data
!apt-get install -q -y libpango-1.0-0 libpangoft2-1.0-0 libharfbuzz0b 2>/dev/null

import torch, multiprocessing
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU:    {torch.cuda.get_device_name(0)}")
    print(f"VRAM:   {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU found — training will be very slow!")
print(f"CPUs:   {multiprocessing.cpu_count()}")

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_DIR = "/content/drive/MyDrive/APS360_checkpoints"
DRIVE_DATA = f"{DRIVE_DIR}/output"
!mkdir -p {DRIVE_DIR}

## 3. Generate training data (or restore from Drive)

**Pipeline:** HTML template → WeasyPrint → PDF → pypdf text → character-aligned BIO labels

This trains the model on the **same noisy PDF text it sees at inference time**.

- Checks Drive first — if a cached dataset exists, restores it (~2 min copy) instead of regenerating (~45 min)
- After generation, saves the full dataset to Drive so future runs are instant
- `--count 150`: 54 templates × 150 = **8,100 training samples**
- `--workers 2`: matches Colab's 2 CPU cores

In [ ]:
import glob, os, shutil

COUNT_PER_TEMPLATE = 200   # was 150; 61 templates × 200 = 12,200 samples
N_TEMPLATES = len(glob.glob("transcript_generator/templates/*.html"))
EXPECTED = N_TEMPLATES * COUNT_PER_TEMPLATE
OUT_DIR = "transcript_generator/output"

def _count_local():
    return len(glob.glob(f"{OUT_DIR}/*/*.json"))

def _count_drive():
    """Count template folders in Drive — os.listdir is reliable on Drive FUSE, glob is not."""
    if not os.path.isdir(DRIVE_DATA):
        return 0, []
    folders = sorted(
        d for d in os.listdir(DRIVE_DATA)
        if os.path.isdir(os.path.join(DRIVE_DATA, d))
    )
    return len(folders), folders

def _copy_drive_to_local(folders):
    """Copy each template folder from Drive to OUT_DIR, printing per-folder progress."""
    os.makedirs(OUT_DIR, exist_ok=True)
    total = len(folders)
    for i, folder in enumerate(folders, 1):
        src = os.path.join(DRIVE_DATA, folder)
        dst = os.path.join(OUT_DIR, folder)
        if os.path.exists(dst):
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
        n_json = len(glob.glob(f"{dst}/*.json"))
        print(f"  [{i:>3}/{total}] {folder}  ({n_json} samples)", flush=True)

def _copy_local_to_drive(folders=None):
    """Save each template folder from OUT_DIR to Drive, printing per-folder progress."""
    os.makedirs(DRIVE_DATA, exist_ok=True)
    if folders is None:
        folders = sorted(
            d for d in os.listdir(OUT_DIR)
            if os.path.isdir(os.path.join(OUT_DIR, d))
        )
    total = len(folders)
    for i, folder in enumerate(folders, 1):
        src = os.path.join(OUT_DIR, folder)
        dst = os.path.join(DRIVE_DATA, folder)
        if os.path.exists(dst):
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
        print(f"  [{i:>3}/{total}] saved {folder}", flush=True)

# ── Main logic ──────────────────────────────────────────────────────────────
print(f"Templates: {N_TEMPLATES}  |  Expected samples: {EXPECTED}")

existing = _count_local()
print(f"Existing local JSON files: {existing}")

if existing >= EXPECTED:
    print("✓ Data already present in runtime — skipping.")

elif os.path.isdir(DRIVE_DATA):
    n_drive_folders, drive_folders = _count_drive()
    print(f"Drive cache: {n_drive_folders} template folders found.")

    if n_drive_folders == 0:
        print("Drive directory exists but is empty — regenerating...")
        generate = True
    else:
        print(f"Restoring {n_drive_folders} folders from Drive...")
        _copy_drive_to_local(drive_folders)
        restored = _count_local()
        print(f"✓ Restored {restored} samples from Drive.")
        generate = restored < EXPECTED
        if generate:
            print(f"Cache incomplete ({restored} < {EXPECTED}) — regenerating all templates...")

    if generate:
        get_ipython().system(f"""python transcript_generator/main.py \
            --all-templates \
            --count {COUNT_PER_TEMPLATE} \
            --pdf-extract \
            --augment-noise \
            --no-pdf \
            --workers 2""")
        print("Saving updated dataset to Drive...")
        _copy_local_to_drive()
        print("✓ Drive cache updated.")

else:
    print(f"No Drive cache found — generating {EXPECTED} transcripts (~55 min)...")
    get_ipython().system(f"""python transcript_generator/main.py \
        --all-templates \
        --count {COUNT_PER_TEMPLATE} \
        --pdf-extract \
        --augment-noise \
        --no-pdf \
        --workers 2""")
    print(f"Saving {N_TEMPLATES} template folders to Drive...")
    _copy_local_to_drive()
    print(f"✓ Dataset saved to {DRIVE_DATA}")


## 4. Train

In [ ]:
import sys
sys.path.insert(0, REPO_DIR)

from model.train import train

history = train(
    data_dir="transcript_generator/output",
    output_dir="model/checkpoints",
    use_fp16=True,   # FP16 on Colab T4
)

## 5. Plot training curves

In [ ]:
import matplotlib.pyplot as plt

epochs = range(1, len(history["train_loss"]) + 1)
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].plot(epochs, history["train_loss"], "o-", label="train")
axes[0].plot(epochs, history["val_loss"],   "s-", label="val")
axes[0].set_title("Loss"); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(epochs, history["val_token_acc"], "o-", color="royalblue")
axes[1].set_title("Val Token Accuracy"); axes[1].grid(alpha=0.3)

axes[2].plot(epochs, history["val_entity_f1"], "o-", color="green")
axes[2].set_title("Val Entity F1"); axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig("training_history.png", dpi=150)
plt.show()

## 6. Save checkpoint to Google Drive

In [ ]:
!rm -rf {DRIVE_DIR}/best
!cp -r model/checkpoints/best              {DRIVE_DIR}/
!cp model/checkpoints/split_info.json      {DRIVE_DIR}/ 2>/dev/null || true
!cp model/logs/history.json                {DRIVE_DIR}/ 2>/dev/null || true
!cp training_history.png                   {DRIVE_DIR}/ 2>/dev/null || true

print(f"✓ Checkpoint saved to {DRIVE_DIR}")

## 7. (Optional) Download checkpoint directly

Alternative to Drive — zip and download straight to your Mac.

In [ ]:
!zip -r bert_checkpoint.zip model/checkpoints/best model/logs/history.json training_history.png

from google.colab import files
files.download("bert_checkpoint.zip")

## 8. (Optional) Smoke test on a test-split sample

In [ ]:
import json, glob
from model.predict import BertNERPredictor
from evaluation.reconstructor import reconstruct_semesters

with open("model/checkpoints/split_info.json") as f:
    split_info = json.load(f)

test_template = split_info["test_templates"][0]
sample_path = sorted(glob.glob(f"transcript_generator/output/{test_template}/*.json"))[0]
with open(sample_path) as f:
    sample = json.load(f)

predictor = BertNERPredictor(checkpoint_dir="model/checkpoints/best")
tags = predictor.predict(sample["tokens"])
semesters = reconstruct_semesters(sample["tokens"], tags)

print(f"Template: {test_template}\n")
for sem in semesters:
    print(f"  {sem['semester_name']}")
    for c in sem["courses"]:
        print(f"    {c['code']:<12} {c['name']:<40} {c['grade']}")